# exp001_tfidf_logreg

This notebook was auto-generated by `src/notebook_gen.py`.
Run all cells to train from scratch and produce `submission.csv`.


In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pyyaml>=6.0', 'tqdm>=4.65', 'scikit-learn>=1.3',
    'pandas>=2.0', 'numpy>=1.24', 'scipy>=1.11',
])


In [ ]:
import os
os.makedirs('src/models', exist_ok=True)


In [ ]:
open('src/__init__.py', 'w').close()  # empty init file


In [ ]:
%%writefile src/data.py
"""
Data loading and fold creation.
Handles both local and Kaggle runtime paths automatically.
"""
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# Searched in order — first match wins
_DATA_DIRS = [
    "/kaggle/input/spr-2026-mammography-report-classification",
    "./spr-2026-mammography-report-classification",
    "../spr-2026-mammography-report-classification",
    "../../spr-2026-mammography-report-classification",
]


def find_data_dir() -> str:
    for d in _DATA_DIRS:
        if os.path.exists(os.path.join(d, "train.csv")):
            return d
    raise FileNotFoundError(
        f"Could not find data directory. Tried: {_DATA_DIRS}\n"
        "Place the competition data in ./spr-2026-mammography-report-classification/"
    )


def load_train(data_dir: str = None) -> pd.DataFrame:
    if data_dir is None:
        data_dir = find_data_dir()
    df = pd.read_csv(os.path.join(data_dir, "train.csv"))
    df.columns = df.columns.str.strip()
    df["target"] = df["target"].astype(int)
    return df


def load_test(data_dir: str = None) -> pd.DataFrame:
    if data_dir is None:
        data_dir = find_data_dir()
    path = os.path.join(data_dir, "test.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"test.csv not found at {path}.\n"
            "This is expected locally — it only exists in the Kaggle evaluation runtime."
        )
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


def make_folds(df: pd.DataFrame, n_folds: int = 5, seed: int = 42) -> pd.DataFrame:
    """Add a 'fold' column (0..n_folds-1) using stratified split on target."""
    df = df.copy().reset_index(drop=True)
    df["fold"] = -1
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fold, (_, val_idx) in enumerate(skf.split(df, df["target"])):
        df.loc[val_idx, "fold"] = fold
    return df


In [ ]:
%%writefile src/features.py
"""
Text preprocessing and TF-IDF feature extraction for Portuguese medical reports.
"""
import pickle
import re
import os
from typing import List, Tuple, Optional

import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer


def clean_text(text: str) -> str:
    """
    Minimal cleaning that preserves medically meaningful tokens.
    - Keeps <DATA> placeholder (correlates with comparative exams → BI-RADS 2/3)
    - Normalises whitespace; lowercases
    - Does NOT strip accents — Portuguese accents carry lexical meaning
    """
    if not isinstance(text, str):
        text = str(text)
    # Collapse newlines / tabs to a single space
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r" {2,}", " ", text).strip()
    text = text.lower()
    return text


def _make_vectorizer(cfg: dict) -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer=cfg.get("analyzer", "word"),
        ngram_range=tuple(cfg.get("ngram_range", [1, 2])),
        max_features=cfg.get("max_features", 150_000),
        sublinear_tf=cfg.get("sublinear_tf", True),
        min_df=cfg.get("min_df", 2),
        strip_accents=None,  # preserve Portuguese characters
    )


def build_features(
    texts_train: List[str],
    texts_val: Optional[List[str]] = None,
    config: Optional[dict] = None,
) -> Tuple:
    """
    Fit TF-IDF (word + char) on texts_train.
    Returns (X_train, vectorizers) or (X_train, X_val, vectorizers).
    """
    cfg = config or {}
    word_cfg = cfg.get("word_tfidf", {})
    char_cfg  = cfg.get("char_tfidf", {})

    # Default char config if not provided
    if "analyzer" not in char_cfg:
        char_cfg = {"analyzer": "char_wb", "ngram_range": [2, 5],
                    "max_features": 150_000, "sublinear_tf": True, "min_df": 3}

    word_vec = _make_vectorizer(word_cfg)
    char_vec  = _make_vectorizer(char_cfg)

    X_word_train = word_vec.fit_transform(texts_train)
    X_char_train  = char_vec.fit_transform(texts_train)
    X_train = hstack([X_word_train, X_char_train], format="csr")

    vectorizers = (word_vec, char_vec)

    if texts_val is not None:
        X_word_val = word_vec.transform(texts_val)
        X_char_val  = char_vec.transform(texts_val)
        X_val = hstack([X_word_val, X_char_val], format="csr")
        return X_train, X_val, vectorizers

    return X_train, vectorizers


def transform_features(texts: List[str], vectorizers: Tuple) -> csr_matrix:
    word_vec, char_vec = vectorizers
    X_word = word_vec.transform(texts)
    X_char  = char_vec.transform(texts)
    return hstack([X_word, X_char], format="csr")


def save_vectorizers(vectorizers: Tuple, out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "vectorizers.pkl"), "wb") as f:
        pickle.dump(vectorizers, f)


def load_vectorizers(out_dir: str) -> Tuple:
    with open(os.path.join(out_dir, "vectorizers.pkl"), "rb") as f:
        return pickle.load(f)


In [ ]:
%%writefile src/evaluate.py
"""
Metrics, reporting, and the master results log.
Always report macro F1 + per-class breakdown — never accuracy alone.
"""
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

CLASSES = [0, 1, 2, 3, 4, 5, 6]
RESULTS_LOG = "experiments/results.csv"

# Columns shown in --compare (focused on rare/hard classes)
_COMPARE_COLS = [
    "experiment", "macro_f1",
    "f1_c0", "f1_c1", "f1_c3", "f1_c4", "f1_c5", "f1_c6",
    "duration_s", "timestamp", "notes",
]


def compute_metrics(y_true, y_pred) -> dict:
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=CLASSES, zero_division=0)
    report = classification_report(
        y_true, y_pred, labels=CLASSES, zero_division=0, output_dict=True
    )
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
    return {"macro_f1": macro_f1, "report": report, "confusion_matrix": cm.tolist()}


def print_metrics(metrics: dict, title: str = "") -> None:
    if title:
        print(f"\n{'='*55}\n{title}\n{'='*55}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}\n")
    print(f"{'Class':<8} {'F1':>6} {'Prec':>6} {'Rec':>6} {'N':>6}")
    print("-" * 38)
    for cls in CLASSES:
        r = metrics["report"].get(str(cls), {})
        f1  = r.get("f1-score",  0.0)
        pre = r.get("precision", 0.0)
        rec = r.get("recall",    0.0)
        n   = int(r.get("support", 0))
        print(f"{cls:<8} {f1:>6.3f} {pre:>6.3f} {rec:>6.3f} {n:>6}")
    print()


def save_metrics(metrics: dict, out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    path = out_dir if out_dir.endswith(".json") else os.path.join(out_dir, "metrics.json")
    with open(path, "w") as f:
        json.dump(metrics, f, indent=2)


def append_to_results_log(
    exp_name: str,
    metrics: dict,
    config: dict,
    timestamp: str = "",
    duration: float = 0.0,
    notes: str = "",
) -> None:
    """Upsert one row into experiments/results.csv."""
    row = {
        "experiment":  exp_name,
        "macro_f1":    round(metrics["macro_f1"], 4),
        "model_type":  config.get("model", {}).get("type", ""),
        "n_folds":     config.get("data", {}).get("n_folds", ""),
        "seed":        config.get("seed", ""),
        "timestamp":   timestamp,
        "duration_s":  round(duration, 1),
        "notes":       notes or config.get("notes", ""),
    }
    for cls in CLASSES:
        r = metrics["report"].get(str(cls), {})
        row[f"f1_c{cls}"] = round(r.get("f1-score", 0.0), 4)

    df_row = pd.DataFrame([row])
    if os.path.exists(RESULTS_LOG):
        existing = pd.read_csv(RESULTS_LOG, keep_default_na=False)
        existing = existing[existing["experiment"] != exp_name]
        df_out = pd.concat([existing, df_row], ignore_index=True)
    else:
        os.makedirs(os.path.dirname(RESULTS_LOG), exist_ok=True)
        df_out = df_row

    df_out = df_out.sort_values("macro_f1", ascending=False)
    df_out.to_csv(RESULTS_LOG, index=False)
    print(f"Results logged → {RESULTS_LOG}")


def print_comparison(full: bool = False) -> None:
    """Print the leaderboard table. full=True shows all columns."""
    if not os.path.exists(RESULTS_LOG):
        print("No experiments logged yet.")
        return

    df = (pd.read_csv(RESULTS_LOG, keep_default_na=False)
            .sort_values("macro_f1", ascending=False)
            .reset_index(drop=True))
    df.index += 1  # rank starts at 1

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 140)
    pd.set_option("display.float_format", "{:.4f}".format)

    print(f"\n{'='*55}")
    print(f"  Experiment leaderboard  (OOF macro F1, descending)")
    print(f"{'='*55}")

    if full:
        print(df.to_string())
    else:
        cols = [c for c in _COMPARE_COLS if c in df.columns]
        print(df[cols].to_string())

    # Highlight the improvement vs the row below
    if len(df) > 1:
        best = df["macro_f1"].iloc[0]
        second = df["macro_f1"].iloc[1]
        print(f"\n  Best vs 2nd: +{best - second:+.4f}  ({df['experiment'].iloc[0]})")
    print()


In [ ]:
%%writefile src/threshold.py
"""
Post-processing: per-class decision threshold tuning on OOF probabilities.
Tuning is explicitly allowed by competition rules. Always tune on OOF only.
"""
import numpy as np
from sklearn.metrics import f1_score

CLASSES = [0, 1, 2, 3, 4, 5, 6]


def tune_thresholds(
    oof_probs: np.ndarray,
    oof_labels: np.ndarray,
    n_iter: int = 500,
    seed: int = 42,
) -> np.ndarray:
    """
    Random-search for per-class additive offsets that maximise OOF macro F1.
    Returns an offset array of shape (n_classes,).
    Apply at inference time: preds = argmax(probs + offsets, axis=1).
    """
    baseline_preds = np.argmax(oof_probs, axis=1)
    best_f1 = f1_score(
        oof_labels, baseline_preds, average="macro", labels=CLASSES, zero_division=0
    )
    best_offsets = np.zeros(len(CLASSES))
    print(f"Threshold tuning baseline macro F1: {best_f1:.4f}")

    rng = np.random.RandomState(seed)
    for _ in range(n_iter):
        offsets = rng.uniform(-0.3, 0.3, size=len(CLASSES))
        preds = np.argmax(oof_probs + offsets, axis=1)
        f1 = f1_score(oof_labels, preds, average="macro", labels=CLASSES, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_offsets = offsets.copy()

    print(f"Threshold tuning best macro F1:     {best_f1:.4f}")
    print(f"Offsets: {dict(zip(CLASSES, best_offsets.round(4)))}")
    return best_offsets


def apply_thresholds(probs: np.ndarray, offsets: np.ndarray) -> np.ndarray:
    return np.argmax(probs + offsets, axis=1)


In [ ]:
%%writefile src/logging_utils.py
"""
Dual-output logging: every print() goes to both the terminal and a log file.
tqdm writes to stderr so it stays on the terminal only — logs stay clean.
"""
import io
import os
import re
import sys
from contextlib import contextmanager
from datetime import datetime


# Strip ANSI escape codes (colour, cursor movement) before writing to file
_ANSI_RE = re.compile(r"\x1b\[[0-9;]*[mABCDEFGHJKSTfhilmnsu]|\r")


class _Tee(io.TextIOBase):
    """Writes to both the original stdout and a log file simultaneously."""

    def __init__(self, log_path: str, original_stdout):
        self._stdout = original_stdout
        self._file = open(log_path, "a", encoding="utf-8", buffering=1)

    def write(self, text: str) -> int:
        self._stdout.write(text)
        self._file.write(_ANSI_RE.sub("", text))
        return len(text)

    def flush(self):
        self._stdout.flush()
        self._file.flush()

    def close(self):
        self._file.close()

    # Needed so tqdm and other libs can check isatty on the underlying stream
    def isatty(self) -> bool:
        return self._stdout.isatty()

    @property
    def encoding(self):
        return self._stdout.encoding


@contextmanager
def run_log(out_dir: str, filename: str = "train.log"):
    """
    Context manager that tees all print() output to out_dir/train.log.
    Usage:
        with run_log(out_dir):
            ... training code ...
    """
    os.makedirs(out_dir, exist_ok=True)
    log_path = os.path.join(out_dir, filename)

    tee = _Tee(log_path, sys.stdout)
    original_stdout = sys.stdout
    sys.stdout = tee
    try:
        print(f"[log] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  →  {log_path}")
        yield log_path
    finally:
        sys.stdout = original_stdout
        tee.close()


In [ ]:
open('src/models/__init__.py', 'w').close()  # empty init file


In [ ]:
%%writefile src/models/linear.py
"""
Sparse linear models: Logistic Regression and LinearSVC (calibrated).
These are the recommended first baseline — fast, interpretable, strong for text.
"""
import os
import pickle

from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC


def build_model(config: dict):
    """
    Build a model from the 'model' section of an experiment config.
    Supported types: logistic_regression, linear_svc
    Both return a sklearn estimator with predict_proba support.
    """
    model_type = config.get("type", "logistic_regression")
    params = config.get("params", {})
    seed = params.get("seed", 42)

    if model_type == "logistic_regression":
        return LogisticRegression(
            C=params.get("C", 1.0),
            max_iter=params.get("max_iter", 2000),
            class_weight=params.get("class_weight", "balanced"),
            solver=params.get("solver", "lbfgs"),
            random_state=seed,
            n_jobs=-1,
        )

    if model_type == "linear_svc":
        # LinearSVC has no predict_proba; wrap with isotonic calibration
        base = LinearSVC(
            C=params.get("C", 1.0),
            max_iter=params.get("max_iter", 2000),
            class_weight=params.get("class_weight", "balanced"),
            random_state=seed,
        )
        return CalibratedClassifierCV(base, method="isotonic", cv=3)

    raise ValueError(
        f"Unknown model type: '{model_type}'. "
        "Choose from: logistic_regression, linear_svc"
    )


def save_model(model, out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "model.pkl"), "wb") as f:
        pickle.dump(model, f)


def load_model(out_dir: str):
    with open(os.path.join(out_dir, "model.pkl"), "rb") as f:
        return pickle.load(f)


In [ ]:
%%writefile src/train.py
"""
OOF cross-validation training loop.
Called by run.py — do not invoke directly.
"""
import os
import shutil
import subprocess
import time
from datetime import datetime

import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm

from src.data import load_train, make_folds
from src.evaluate import (
    append_to_results_log,
    compute_metrics,
    print_metrics,
    save_metrics,
)
from src.features import (
    build_features,
    clean_text,
    save_vectorizers,
)
from src.logging_utils import run_log
from src.models.linear import build_model, save_model
from src.threshold import tune_thresholds

CLASSES = [0, 1, 2, 3, 4, 5, 6]


def _git_hash() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "unknown"


def _probs_to_matrix(probs, model_classes):
    """Map model.predict_proba output to a (n, 7) matrix aligned to CLASSES."""
    matrix = np.zeros((len(probs), len(CLASSES)))
    for i, cls in enumerate(model_classes):
        matrix[:, CLASSES.index(int(cls))] = probs[:, i]
    return matrix


def _step(pbar, msg):
    pbar.set_postfix_str(msg, refresh=True)


def run_training(config_path: str, notes_override: str = None) -> tuple:
    with open(config_path) as f:
        config = yaml.safe_load(f)

    exp_name = config["experiment_name"]
    out_dir  = os.path.join("experiments", exp_name)
    os.makedirs(out_dir, exist_ok=True)

    dest = os.path.join(out_dir, "config.yaml")
    if os.path.abspath(config_path) != os.path.abspath(dest):
        shutil.copy(config_path, dest)

    # Notes: CLI override > config field > empty
    notes = notes_override or config.get("notes", "")

    with run_log(out_dir):
        _run(config, exp_name, out_dir, config_path, notes)

    return out_dir


def _run(config, exp_name, out_dir, config_path, notes):
    t_start = time.time()
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    seed      = config.get("seed", 42)
    n_folds   = config["data"]["n_folds"]
    feat_cfg  = config.get("features", {})
    model_cfg = config["model"]

    print(f"\n{'='*55}")
    print(f"Experiment : {exp_name}")
    print(f"Config     : {config_path}")
    print(f"Started    : {timestamp}")
    print(f"Git commit : {_git_hash()}")
    if notes:
        print(f"Notes      : {notes}")
    print(f"{'='*55}")

    # ── Load & prepare data ──────────────────────────────
    print("Loading data...", end=" ", flush=True)
    df = load_train()
    df["text"] = df["report"].apply(clean_text)
    df = make_folds(df, n_folds=n_folds, seed=seed)
    print(f"done  ({len(df):,} rows, {n_folds} folds)")

    oof_probs = np.zeros((len(df), len(CLASSES)))
    fold_f1s  = []

    # ── OOF cross-validation ─────────────────────────────
    fold_bar = tqdm(range(n_folds), desc="CV folds", unit="fold", ncols=70)
    for fold in fold_bar:
        train_mask = df["fold"] != fold
        val_mask   = df["fold"] == fold
        val_idx    = df.index[val_mask].tolist()

        n_tr = train_mask.sum()
        n_val = val_mask.sum()
        fold_bar.set_description(f"Fold {fold+1}/{n_folds}  ({n_tr}tr/{n_val}val)")

        _step(fold_bar, "vectorizing")
        X_train, X_val, vectorizers = build_features(
            df.loc[train_mask, "text"].tolist(),
            df.loc[val_mask,   "text"].tolist(),
            feat_cfg,
        )
        y_train = df.loc[train_mask, "target"].values
        y_val   = df.loc[val_mask,   "target"].values

        _step(fold_bar, "fitting")
        model = build_model(model_cfg)
        model.fit(X_train, y_train)

        _step(fold_bar, "scoring")
        val_probs   = model.predict_proba(X_val)
        prob_matrix = _probs_to_matrix(val_probs, model.classes_)
        for i, pos in enumerate(val_idx):
            oof_probs[pos] = prob_matrix[i]

        fold_preds   = np.argmax(prob_matrix, axis=1)
        fold_metrics = compute_metrics(y_val, fold_preds)
        fold_f1s.append(fold_metrics["macro_f1"])
        fold_bar.set_postfix({"F1": f"{fold_metrics['macro_f1']:.4f}"}, refresh=True)

    fold_bar.close()

    # ── OOF evaluation ───────────────────────────────────
    oof_preds   = np.argmax(oof_probs, axis=1)
    oof_metrics = compute_metrics(df["target"].values, oof_preds)
    print_metrics(oof_metrics, title=f"OOF Results — {exp_name}")
    print(f"Fold F1s : {[round(f, 4) for f in fold_f1s]}")
    print(f"Mean ± σ : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

    # Save OOF predictions (used for threshold tuning and error analysis)
    oof_df = df[["ID", "target", "fold"]].copy()
    oof_df["oof_pred"] = oof_preds
    for i, cls in enumerate(CLASSES):
        oof_df[f"p{cls}"] = oof_probs[:, i]
    oof_df.to_csv(os.path.join(out_dir, "oof_preds.csv"), index=False)
    save_metrics(oof_metrics, out_dir)

    # ── Optional threshold tuning ────────────────────────
    if config.get("threshold", {}).get("enabled", False):
        print("\n--- Threshold Tuning ---")
        offsets = tune_thresholds(oof_probs, df["target"].values, seed=seed)
        np.save(os.path.join(out_dir, "thresholds.npy"), offsets)

        tuned_preds   = np.argmax(oof_probs + offsets, axis=1)
        tuned_metrics = compute_metrics(df["target"].values, tuned_preds)
        print_metrics(tuned_metrics, title="OOF Results (after threshold tuning)")
        save_metrics(tuned_metrics, os.path.join(out_dir, "metrics_tuned.json"))
        oof_metrics = tuned_metrics

    # ── Retrain on full data ─────────────────────────────
    print("\n--- Retraining on full data ---")
    with tqdm(total=3, desc="Full retrain", unit="step", ncols=70) as pbar:
        pbar.set_postfix_str("vectorizing", refresh=True)
        X_full, vectorizers_full = build_features(df["text"].tolist(), config=feat_cfg)
        pbar.update(1)

        pbar.set_postfix_str("fitting", refresh=True)
        model_full = build_model(model_cfg)
        model_full.fit(X_full, df["target"].values)
        pbar.update(1)

        pbar.set_postfix_str("saving", refresh=True)
        save_model(model_full, out_dir)
        save_vectorizers(vectorizers_full, out_dir)
        pbar.update(1)

    duration = time.time() - t_start
    print(f"\nDuration   : {duration:.0f}s  ({duration/60:.1f}m)")
    append_to_results_log(exp_name, oof_metrics, config,
                          timestamp=timestamp, duration=duration, notes=notes)
    print(f"Outputs    : experiments/{exp_name}/")


In [ ]:
%%writefile src/predict.py
"""
Generate submission.csv from a trained experiment directory.
Called by run.py — do not invoke directly.
"""
import os

import numpy as np
import pandas as pd

from src.data import load_test
from src.features import clean_text, load_vectorizers, transform_features
from src.models.linear import load_model

CLASSES = [0, 1, 2, 3, 4, 5, 6]


def run_predict(exp_dir: str) -> str:
    test = load_test()
    test["text"] = test["report"].apply(clean_text)

    vectorizers = load_vectorizers(exp_dir)
    model = load_model(exp_dir)

    X_test = transform_features(test["text"].tolist(), vectorizers)
    probs = model.predict_proba(X_test)

    # Align probability columns to CLASSES order
    prob_matrix = np.zeros((len(test), len(CLASSES)))
    for i, cls in enumerate(model.classes_):
        prob_matrix[:, CLASSES.index(int(cls))] = probs[:, i]

    # Apply thresholds if they were tuned for this experiment
    threshold_path = os.path.join(exp_dir, "thresholds.npy")
    if os.path.exists(threshold_path):
        offsets = np.load(threshold_path)
        preds = np.argmax(prob_matrix + offsets, axis=1)
        print("Applied saved threshold offsets.")
    else:
        preds = np.argmax(prob_matrix, axis=1)

    sub = pd.DataFrame({"ID": test["ID"], "target": preds})
    sub_path = os.path.join(exp_dir, "submission.csv")
    sub.to_csv(sub_path, index=False)

    print(f"Submission saved → {sub_path}  ({len(sub)} rows)")
    print(f"Prediction distribution:\n{sub['target'].value_counts().sort_index().to_string()}")
    return sub_path


In [ ]:
import os, shutil
os.makedirs('configs', exist_ok=True)
open('configs/exp001_tfidf_logreg.yaml', 'w').write("# Experiment 001 \u2014 TF-IDF (word + char) + Logistic Regression\n# Baseline. Expected OOF macro F1: ~0.55\u20130.65 depending on class 5/6 luck.\n\nexperiment_name: exp001_tfidf_logreg\n\nseed: 42\n\ndata:\n  n_folds: 5\n\nfeatures:\n  word_tfidf:\n    analyzer: word\n    ngram_range: [1, 2]\n    max_features: 150000\n    sublinear_tf: true\n    min_df: 2\n  char_tfidf:\n    analyzer: char_wb\n    ngram_range: [2, 5]\n    max_features: 150000\n    sublinear_tf: true\n    min_df: 3\n\nmodel:\n  type: logistic_regression\n  params:\n    C: 1.0\n    max_iter: 2000\n    class_weight: balanced\n    solver: lbfgs\n    multi_class: multinomial\n    seed: 42\n\nthreshold:\n  enabled: false\n")

from src.train import run_training
from src.predict import run_predict
exp_dir = run_training('configs/exp001_tfidf_logreg.yaml')
run_predict(exp_dir)

# Kaggle requires submission.csv at the working directory root
shutil.copy(os.path.join(exp_dir, 'submission.csv'), 'submission.csv')
print('submission.csv copied to', os.getcwd())


In [ ]:
import pandas as pd, os
sub_path = os.path.join(exp_dir, 'submission.csv')
sub = pd.read_csv(sub_path)
assert {'ID', 'target'}.issubset(sub.columns), f'Bad columns: {sub.columns.tolist()}'
assert sub['target'].between(0, 6).all(), 'Targets out of range 0-6'
print(f'submission.csv looks good: {len(sub)} rows')
print(sub['target'].value_counts().sort_index().to_string())
sub.head()
